In [17]:
import pandas as pd
final = pd.read_parquet("../data/processed/modeling_table.parquet")
final_clean = final.dropna(subset=["is_delayed"]).copy()

In [18]:
majority_baseline = final_clean["is_delayed"].value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

Majority-class baseline accuracy: 0.721


In [19]:
route_rates = final_clean.groupby("route_short_name")["is_delayed"].transform("mean")
route_baseline_preds = (route_rates > 0.5).astype(int)
route_baseline_acc = (route_baseline_preds == final_clean["is_delayed"]).mean()
print(f"Route-based naive baseline accuracy: {route_baseline_acc:.3f}")

Route-based naive baseline accuracy: 0.724


In [20]:
final = pd.read_parquet("../data/processed/modeling_table.parquet")
final_clean = final.dropna(subset=["is_delayed"]).copy()

majority_baseline = final_clean["is_delayed"].value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

route_rates = final_clean.groupby("route_short_name")["is_delayed"].transform("mean")
route_baseline_preds = (route_rates > 0.5).astype(int)
route_baseline_acc = (route_baseline_preds == final_clean["is_delayed"]).mean()
print(f"Majority baseline: {majority_baseline:.4f}")
print(f"Route baseline: {route_baseline_acc:.4f}")
print(f"Fraction of rows flagged 'predicted delayed': {(route_rates > 0.5).mean():.4f}")
print(final_clean.groupby('route_short_name')['is_delayed'].mean().describe())

Majority-class baseline accuracy: 0.721
Majority baseline: 0.7207
Route baseline: 0.7237
Fraction of rows flagged 'predicted delayed': 0.0320
count    577.000000
mean       0.304363
std        0.144670
min        0.004827
25%        0.212190
50%        0.299718
75%        0.378194
max        1.000000
Name: is_delayed, dtype: float64


In [22]:
print((final_clean.groupby('route_short_name')['is_delayed'].mean() > 0.5).sum(), "routes now cross 50%")

49 routes now cross 50%


Baselines (23-date winter dataset): majority-class baseline is 72.1% accuracy (all-not-delayed) — winter-only delay rate is 27.9%, higher than the mixed 5-date sample's 22.3%, consistent with Phase 3's finding that winter trips run meaningfully more delayed than summer. A naive route-based baseline (predict delayed if a route's historical rate > 50%) reaches 72.4% — barely better than majority, and the underlying reason is the same as in the smaller sample: even with winter's higher delay rate, per-route rates still cluster well under 50% (mean 30.4%, 75th percentile 37.8%), with only 49 of 577 routes crossing the threshold. This confirms route needs to be used as a continuous/weighted signal in a real model, not a hard cutoff — the EDA finding (11-23x rate gap between metro and bus) is real, but no fixed threshold captures it well.

In [21]:
final = pd.read_parquet("../data/processed/modeling_table.parquet")
print(final["service_date"].nunique(), "dates,", len(final), "rows")
print(final["temperature_c"].describe())
print(final["is_delayed"].mean())

23 dates, 13283093 rows
count    1.328309e+07
mean    -1.967149e+00
std      5.081611e+00
min     -1.420000e+01
25%     -4.800000e+00
50%     -7.000000e-01
75%      1.500000e+00
max      7.500000e+00
Name: temperature_c, dtype: float64
0.279327824779203
